In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from losses.final_loss import InverseDistanceBoundaryDiceLoss
from models.cbam_nnunet import nnUNet3D_CBAM
from utils.unets_helper_functions import (
    set_seed,
    save_checkpoint,
    final_PatchDataset_cbam,
    evaluate_full_volume_cbam,
    compute_per_class_dice
    
)

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 12
FOREGROUND_PROB = 0.7
NUM_WORKERS = 2
train_dataset = final_PatchDataset_cbam(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=True)

val_dataset = final_PatchDataset_cbam(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=NUM_WORKERS,    
    pin_memory=True
)

In [6]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24 
).to(device)

print("cbam nnU-Net style model Initialized")


cbam nnU-Net style model Initialized


In [8]:
EPOCHS = 40

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

weights = torch.tensor([0.05, 1, 1.3, 1.8, 2, 2, 1]).to(device)

loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.5,
    class_weights = weights
)

scaler = torch.cuda.amp.GradScaler()

C:\Users\dhanu\AppData\Local\Temp\ipykernel_13616\2878956367.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [7]:
def train_one_epoch_cbam(model, loader, optimizer, scaler, loss_fn, device):

    model.train()
    total_loss = 0

    for images, labels, dist_maps in tqdm(loader):

        images = images.to(device)
        labels = labels.long().to(device)
        dist_maps = dist_maps.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(images)

            # Deep supervision unpack
            out, ds2, ds3, ds4 = outputs


            loss_main = loss_fn(out, labels, dist_maps)
            loss_ds2  = loss_fn(ds2, labels, dist_maps)
            loss_ds3  = loss_fn(ds3, labels, dist_maps)
            loss_ds4  = loss_fn(ds4, labels, dist_maps)

            loss = (
                1.0 * loss_main +
                0.5 * loss_ds2 +
                0.25 * loss_ds3 +
                0.125 * loss_ds4
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate_one_epoch_cbam(model, loader, loss_fn, device, num_classes=7):

    model.eval()
    total_loss = 0
    all_dices = []

    with torch.no_grad():
        for images, labels, dist_maps in loader:

            images = images.to(device)
            labels = labels.long().to(device)
            dist_maps = dist_maps.to(device)

            with torch.amp.autocast("cuda"):

                outputs = model(images)
                out = outputs[0]

                loss = loss_fn(out, labels, dist_maps)

            total_loss += loss.item()

            batch_dice = compute_per_class_dice(out, labels, num_classes)
            all_dices.append(batch_dice)

    mean_loss = total_loss / len(loader)

    all_dices = np.array(all_dices)
    mean_dices = np.mean(all_dices, axis=0)

    return mean_loss, mean_dices

In [ ]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "custom_loss_cbam")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_combined = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Organs which are hard to segment (S=3, L=4, R=5)
    small_org_score = (val_dice[2] + val_dice[3] + val_dice[4]) / 3
    # Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * small_org_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "05_sc_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "05_sc_best_dice_model.pth")
        )
        print("Saved Best Dice Model")



    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "05_sc_best_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "05_sc_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()



100%|██████████| 198/198 [20:59<00:00,  6.36s/it]



Epoch 0
Train Loss: 0.8743
Val Loss: 0.3150
Per Class Dice: [0.68160364 0.22222222 0.06618508 0.34559433 0.03657885 0.62328689]
Mean Dice: 0.2588
Combined Score: 0.2260
Saved Best Loss Model
Saved Best Dice Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [17:19<00:00,  5.25s/it]



Epoch 1
Train Loss: 0.5369
Val Loss: 0.1558
Per Class Dice: [0.80717825 0.54138978 0.49459072 0.66444257 0.31025923 0.84144035]
Mean Dice: 0.5704
Combined Score: 0.5462
Saved Best Loss Model
Saved Best Dice Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 198/198 [14:43<00:00,  4.46s/it]


...

100%|██████████| 198/198 [08:34<00:00,  2.60s/it]



Epoch 26
Train Loss: 0.2232
Val Loss: 0.1051
Per Class Dice: [0.61252013 0.78445976 0.83689683 0.6374882  0.74706589 0.80400129]
Mean Dice: 0.7620
Combined Score: 0.7555
LR: 0.000055


100%|██████████| 198/198 [10:05<00:00,  3.06s/it]



Epoch 27
Train Loss: 0.2273
Val Loss: 0.0501
Per Class Dice: [0.95232803 0.89478796 0.86442515 0.91820218 0.98115047 0.93197272]
Mean Dice: 0.9181
Combined Score: 0.9191
Saved Best Loss Model
Saved Best Dice Model
Saved Best Combined Model
LR: 0.000048


  5%|▍         | 9/198 [00:30<07:16,  2.31s/it]

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24
).to(device)

model_path = "../experiments/custom_loss_cbam/05_sc_best_combined_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\dhanu\AppData\Local\Temp\ipykernel_18608\3275529065.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


In [ ]:
import nibabel as nib
def get_gaussian_weight_map(patch_size):
    import numpy as np

    ranges = [np.linspace(-1, 1, s) for s in patch_size]
    z, y, x = np.meshgrid(*ranges, indexing="ij")

    dist = z**2 + y**2 + x**2
    gaussian = np.exp(-dist / 0.5)

    gaussian = gaussian / np.max(gaussian)
    return torch.tensor(gaussian, dtype=torch.float32)


def final_sliding_window_cbam(model, image, patch_size=96, stride=48, device="cuda"):
    model.eval()

    image_t = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    _, _, D, H, W = image_t.shape
    num_classes = 7

    # CPU accumulation (safe for memory)
    output     = torch.zeros((1, num_classes, D, H, W), dtype=torch.float32)
    weight_map = torch.zeros_like(output)

    gaussian = get_gaussian_weight_map((patch_size, patch_size, patch_size))
    gaussian_gpu = gaussian.unsqueeze(0).unsqueeze(0).to(device)
    gaussian_cpu = gaussian.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        z_steps = sorted(set(list(range(0, max(1, D - patch_size), stride)) + [max(0, D - patch_size)]))
        y_steps = sorted(set(list(range(0, max(1, H - patch_size), stride)) + [max(0, H - patch_size)]))
        x_steps = sorted(set(list(range(0, max(1, W - patch_size), stride)) + [max(0, W - patch_size)]))

        print(f"Volume: {D}x{H}x{W} | Patches: {len(z_steps)*len(y_steps)*len(x_steps)} | Stride: {stride}")

        for z in z_steps:
            for y in y_steps:
                for x in x_steps:

                    patch = image_t[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size]

                    # ---- Forward ----
                    logits = model(patch)[0]   # deep supervision → take main output
                    probs = torch.softmax(logits, dim=1)

                    # ---- Weight with Gaussian ----
                    probs = probs * gaussian_gpu

                    # ---- Move to CPU once ----
                    probs_cpu = probs.cpu()

                    output[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size] += probs_cpu
                    weight_map[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size] += gaussian_cpu

    # ---- Normalize ----
    output = output / weight_map.clamp(min=1e-6)

    # ---- Argmax ----
    pred = torch.argmax(output, dim=1).squeeze().numpy()

    return pred


def final_evaluate_full_volume_cbam(model, cases, images_dir, labels_dir,
                             patch_size=96, stride=48, device="cuda"):

    model.eval()
    all_dices = []

    for case in cases:

        image = nib.load(f"{images_dir}/{case}.nii.gz").get_fdata(dtype=np.float32)
        label = nib.load(f"{labels_dir}/{case}.nii.gz").get_fdata().astype(np.uint8)

        pred = final_sliding_window_cbam(
            model,
            image,
            patch_size=patch_size,
            stride=stride,
            device=device
        )

        print(f"\nCase: {case}")
        print("Pred labels:", np.unique(pred))
        print("GT labels:", np.unique(label))

        case_dices = []

        for cls in range(1, 7):  # ignore background

            pred_cls = (pred == cls)
            label_cls = (label == cls)

            intersection = np.sum(pred_cls & label_cls)
            union = np.sum(pred_cls) + np.sum(label_cls)

            # ---- SAFE Dice ----
            if union == 0:
                dice = 1.0  # both empty → perfect
            else:
                dice = (2 * intersection) / union

            case_dices.append(dice)

        all_dices.append(case_dices)

        print(f"Dice: {np.round(case_dices, 4)}")

    all_dices = np.array(all_dices)

    mean_dice = np.mean(all_dices, axis=0)
    std_dice  = np.std(all_dices, axis=0)

    print("\n===== FINAL RESULTS =====")
    print("Mean Dice:", np.round(mean_dice, 4))
    print("Std Dice :", np.round(std_dice, 4))

    return mean_dice, std_dice



In [ ]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.9288667444267819), np.float64(0.8498833723343191), np.float64(0.8225405925070858), np.float64(0.8635864685524534), np.float64(0.8488246712865811), np.float64(0.921776143964056)]
Volume: 243x383x383 | Patches: 245 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.9194803640087761), np.float64(0.781206657609911), np.float64(0.7222222227232644), np.float64(0.8261164989646572), np.float64(0.838892640216818), np.float64(0.9003646291918228)]
Volume: 264x333x333 | Patches: 180 | Stride: 48
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.9161690629762255), np.float64(0.8190537601200502), np.float64(0.8507666101349802), np.float64(0.842786958580626), np.float64(0.8077244609532482), np.float64(0.9022222980399737)]
Volume: 248x395x395 | Patches